# 27 · Seeds and the BiLSTM on the clean splits

Runs two protocols that were locked before any training, on a Colab **GPU** runtime
(Runtime → Change runtime type → T4 GPU), in about **1½ hours**:

- [`docs/experiments/bilstm_clean_splits/PROTOCOL.md`](../docs/experiments/bilstm_clean_splits/PROTOCOL.md)
  (version 3): the BiLSTM on the three clean views with seed 42, plus seeds 43 and 44 on the
  out-of-time view; and the four models timed on one CPU for the measured frontier.
- [`docs/experiments/distilbert_seed_robustness/PROTOCOL.md`](../docs/experiments/distilbert_seed_robustness/PROTOCOL.md):
  the DistilBERT full fine-tune and scratch LoRA with seeds 43 and 44 on the out-of-time view.

It checks out the pinned code commit, rebuilds the splits and stops unless they are the ones
Naive Bayes was scored on. It applies both protocols' rules with
`scripts/compare_bilstm_clean.py` and `scripts/compare_seeds.py`.

**If Colab disconnects:** with `USE_DRIVE = True`, each finished run is kept in Google Drive.
Run the notebook again from the top; finished runs are skipped.

**Send back** `seeds_and_bilstm_results.zip`. It holds metrics, logits, logs, timings and the
comparisons, not model weights.

In [ ]:
from pathlib import Path

SOURCE_COMMIT = "ed4070c72c452d8e9ce80568e30170e0cafdd32e"
# Keep each finished run in Google Drive (MyDrive/hotel-review-nlp-seeds), so a disconnect
# loses only the run in progress. Colab asks for permission once.
USE_DRIVE = True
REPO = Path("/content/hotel-review-nlp")
CONFIG_DIR = Path("runs/configs")  # the seed configs derived below
EXTRA_SEEDS = (43, 44)  # fixed by both protocols before any run

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then run again from the top.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import subprocess
import sys

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/krimits/hotel-review-nlp.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_COMMIT], check=True)
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if GIT_COMMIT != SOURCE_COMMIT:
    raise RuntimeError(f"checked out {GIT_COMMIT}, not the pinned {SOURCE_COMMIT}")
sys.path.insert(0, str(REPO / "src"))
# The transformers version of the phase 2 runs and of the CI.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "transformers==4.56.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "."], check=True)
print("commit", GIT_COMMIT)

## 1. Data: the splits Naive Bayes was scored on

In [ ]:
import json

from reviewnlp.utils.experiments import fingerprint_splits
from reviewnlp.utils.seed import load_config


def run(command: str) -> None:
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True)


VIEWS = {"time": "configs/baselines_time.yaml", "random": "configs/baselines.yaml",
         "hotel": "configs/baselines_hotel.yaml"}
views = json.loads(Path("docs/experiments/results/split_views.json").read_text(encoding="utf-8"))

run("python scripts/fetch_booking_515k.py")
for view, data_config in VIEWS.items():
    processed = load_config(data_config)["data"]["processed_dir"]
    if not Path(processed, "test.parquet").exists():
        run(f"python -m reviewnlp.data.preprocess --config {data_config}")
    if fingerprint_splits(processed) != views[view]["splits"]:
        raise RuntimeError(f"The {view} split differs from the one Naive Bayes was scored on. Stopping.")
    print(f"{view}: the same reviews, in the same order, as in split_views.json")

## 2. The runs

The order puts what decides first: the seed-42 BiLSTM on the out-of-time view (the BiLSTM
protocol's primary comparison), then the seeds, then the BiLSTM on the other two views.

In [ ]:
import yaml

LORA = ["--r", "8", "--alpha", "16", "--dropout", "0.0", "--lr", "1e-4"]
BILSTM = "reviewnlp.baselines.bilstm"
RUNS = {}  # (view, name) -> (module, config, output directory, extra arguments)


def derived(base: str, seed: int, name: str) -> str:
    """A copy of a committed config in which only the seed and the output directory change."""
    config = load_config(base)
    config["seed"] = seed
    config["output"]["model_dir"] = f"runs/{name}"
    path = CONFIG_DIR / f"{name}.yaml"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    return str(path)


RUNS["time", "bilstm"] = (BILSTM, "configs/bilstm_v2_time.yaml", "runs/bilstm_v2_time", [])
for seed in EXTRA_SEEDS:
    name = f"bilstm_v2_time_seed{seed}"
    RUNS["time", f"bilstm_seed{seed}"] = (BILSTM, derived("configs/bilstm_v2_time.yaml", seed, name), f"runs/{name}", [])
for seed in EXTRA_SEEDS:
    name = f"distilbert_v2_time_seed{seed}"
    config = derived("configs/distilbert_v2_time.yaml", seed, name)
    RUNS["time", f"distilbert_seed{seed}"] = ("reviewnlp.llm.train_distilbert", config, f"runs/{name}", [])
    RUNS["time", f"distilbert_lora_scratch_seed{seed}"] = (
        "reviewnlp.llm.train_distilbert_lora", config, f"runs/{name}_lora_scratch", LORA)
RUNS["random", "bilstm"] = (BILSTM, "configs/bilstm_v2.yaml", "runs/bilstm_v2", [])
RUNS["hotel", "bilstm"] = (BILSTM, "configs/bilstm_v2_hotel.yaml", "runs/bilstm_v2_hotel", [])
for (view, name), (_, config, out_dir, _) in RUNS.items():
    print(f"{view:6} {name:32} {config} -> {out_dir}")

## 3. Training (about 70 minutes on a T4)

Each BiLSTM takes a few minutes; each DistilBERT full fine-tune about 15 and each LoRA about 10.
A run whose `metrics.json` already exists is skipped.

In [ ]:
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    kept = Path("/content/drive/MyDrive/hotel-review-nlp-seeds")
    for _, _, out_dir, _ in RUNS.values():
        (kept / Path(out_dir).name).mkdir(parents=True, exist_ok=True)
        Path(out_dir).parent.mkdir(parents=True, exist_ok=True)
        if not Path(out_dir).exists() and not Path(out_dir).is_symlink():
            Path(out_dir).symlink_to(kept / Path(out_dir).name)


def train(view: str, name: str, module: str, config: str, out_dir: str, extra: list[str]) -> None:
    out = Path(out_dir)
    if (out / "metrics.json").exists():
        print(f"{view}/{name}: already trained, kept")
    else:
        out.mkdir(parents=True, exist_ok=True)
        request = {"commit": GIT_COMMIT, "gpu": torch.cuda.get_device_name(0), "module": module,
                   "config": config, "extra_args": extra}
        (out / "request.json").write_text(json.dumps(request, indent=1) + "\n", encoding="utf-8")
        command = [sys.executable, "-u", "-m", module, "--config", config, *extra]
        print("$", " ".join(command), flush=True)
        with open(out / "train.log", "w", encoding="utf-8") as log, subprocess.Popen(
                command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
        if process.returncode:
            raise RuntimeError(f"{view}/{name} failed; see {out / 'train.log'}")
    metrics = json.loads((out / "metrics.json").read_text(encoding="utf-8"))
    if metrics["data"] != views[view]["splits"]:
        raise RuntimeError(f"{view}/{name} was trained or scored on other data")
    print(f"{view}/{name}: best dev macro-F1 {metrics['best_dev_macro_f1']:.4f}, "
          f"{metrics['training_seconds'] / 60:.1f} minutes")


for (view, name), (module, config, out_dir, extra) in RUNS.items():
    train(view, name, module, config, out_dir, extra)

## 4. The measured frontier (about 10 minutes)

Naive Bayes is trained again on the out-of-time view, only to be timed. The four models are
timed on one CPU thread, one review at a time. DistilBERT is timed with the seed-43
checkpoints of this session: the architecture, not the weights, sets the time. If this step
fails, the next one still applies both rules, without the frontier.

In [ ]:
LATENCY = Path("docs/experiments/results/bilstm_v2/latency_frontier.json")
try:
    run("python -m reviewnlp.baselines.classical --config configs/baselines_time.yaml")
    run(f"python scripts/benchmark_frontier.py --bilstm {RUNS['time', 'bilstm'][2]} "
        f"--distilbert {RUNS['time', 'distilbert_seed43'][2]} "
        f"--lora {RUNS['time', 'distilbert_lora_scratch_seed43'][2]} --out {LATENCY}")
except subprocess.CalledProcessError:
    print("The timing step failed (see above). Go on: the rules are applied without the frontier.")

## 5. Apply the rules, collect and download

In [ ]:
import hashlib
import platform
import shutil
import zipfile
from importlib.metadata import version

KEEP = ["metrics.json", "test_logits.npy", "test_labels.npy", "dev_logits.npy", "dev_labels.npy",
        "request.json", "lora_scratch_config.json", "run_config.yaml", "train_log.txt"]
RESULTS = Path("docs/experiments/results")
targets = []
for (view, name), (module, config, out_dir, _) in RUNS.items():
    bundle = "bilstm_v2" if module == BILSTM else "distilbert_v2"
    target = RESULTS / bundle / view / name
    target.mkdir(parents=True, exist_ok=True)
    for file in KEEP:
        if Path(out_dir, file).exists():
            shutil.copy2(Path(out_dir, file), target / file)
    if not (target / "train_log.txt").exists() and Path(out_dir, "train.log").exists():
        shutil.copy2(Path(out_dir, "train.log"), target / "train_log.txt")
    if not (target / "run_config.yaml").exists():
        shutil.copy2(config, target / "run_config.yaml")
    targets.append(target)

latency = ["--latency", str(LATENCY)] if LATENCY.exists() else []
for command in ([sys.executable, "scripts/compare_bilstm_clean.py", *latency],
                [sys.executable, "scripts/compare_seeds.py"]):
    print("$", " ".join(command), flush=True)
    if subprocess.run(command).returncode:
        print("This comparison failed (see above). The results are still zipped, so it can be checked.")

environment = {
    "commit": GIT_COMMIT, "gpu": torch.cuda.get_device_name(0), "python": platform.python_version(),
    "packages": {name: version(name) for name in
                 ("torch", "transformers", "tokenizers", "scikit-learn", "numpy", "pandas")},
    "extra_seeds": list(EXTRA_SEEDS)}
(RESULTS / "bilstm_v2" / "environment.json").write_text(json.dumps(environment, indent=1) + "\n", encoding="utf-8")

collected = sorted({path for target in targets for path in target.rglob("*") if path.is_file()}
                   | {path for path in (RESULTS / "bilstm_v2").glob("*.json")}
                   | {path for path in [RESULTS / "distilbert_seed_robustness.json"] if path.exists()})
manifest = {"commit": GIT_COMMIT,
            "files": [{"path": path.as_posix(), "bytes": path.stat().st_size,
                       "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
                      for path in collected if path.name != "artifact_manifest.json"]}
(RESULTS / "bilstm_v2" / "artifact_manifest.json").write_text(json.dumps(manifest, indent=1) + "\n", encoding="utf-8")
with zipfile.ZipFile("seeds_and_bilstm_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(set(collected) | {RESULTS / "bilstm_v2" / "artifact_manifest.json"}):
        archive.write(path, path.as_posix())
print(f"{len(manifest['files']) + 1} files")

In [ ]:
from google.colab import files

files.download("seeds_and_bilstm_results.zip")

## 6. After the run

Send `seeds_and_bilstm_results.zip` back. Its paths are the repository's: the bundle is
checked against `artifact_manifest.json` and the commit, then each protocol's rule is
recorded exactly as it fired, in a decision note. If a step failed, the zip shows the last
one that finished.